# Bare `como` spaCy Filter Experiment

Isolated experiment for expanding beyond spec 0006 without running spaCy over the full corpus.

Pipeline:

```text
bronze corpus
  -> Spark regex prefilter for bare como
  -> Python candidate extraction with bare_como_curated_ground_v2
  -> local spaCy classification only on candidate windows
  -> NLP-filtered artifacts under outputs/experiments/bare_como_spacy_filter
```

This notebook does not write main evaluation outputs.


In [1]:
import os
from pathlib import Path
import sys


def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "src/tal_qual").exists():
            return candidate
        if (candidate / "work/src/tal_qual").exists():
            return candidate / "work"
    raise RuntimeError(f"Could not find repository root from {start}")


REPO_ROOT = find_repo_root(Path.cwd())
os.chdir(REPO_ROOT)
SRC_DIR = REPO_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

REPO_ROOT


PosixPath('/home/jovyan/work')

In [2]:
import tempfile
import zipfile

from pyspark.sql import SparkSession
from pyspark.sql import functions as F

SPARK_MASTER = os.environ.get("TAL_QUAL_SPARK_MASTER", "local[4]")
SPARK_PARALLELISM = os.environ.get("TAL_QUAL_SPARK_PARALLELISM", "4")
SPARK_SHUFFLE_PARTITIONS = os.environ.get("TAL_QUAL_SPARK_SHUFFLE_PARTITIONS", "4")
SPARK_DRIVER_MEMORY = os.environ.get("TAL_QUAL_SPARK_DRIVER_MEMORY", "4g")

spark = (
    SparkSession.builder
    .master(SPARK_MASTER)
    .appName("tal-qual-bare-como-spacy-filter")
    .config("spark.driver.memory", SPARK_DRIVER_MEMORY)
    .config("spark.default.parallelism", SPARK_PARALLELISM)
    .config("spark.sql.shuffle.partitions", SPARK_SHUFFLE_PARTITIONS)
    .config("spark.sql.files.maxPartitionBytes", str(128 * 1024 * 1024))
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.sql.adaptive.coalescePartitions.enabled", "true")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

package_zip = Path(tempfile.gettempdir()) / "tal_qual_src.zip"
with zipfile.ZipFile(package_zip, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for file_path in sorted((SRC_DIR / "tal_qual").rglob("*.py")):
        archive.write(file_path, file_path.relative_to(SRC_DIR))
spark.sparkContext.addPyFile(str(package_zip))

SPARK_MASTER, spark.version


('local[4]', '4.1.1')

In [3]:
import pandas as pd
import spacy

from tal_qual.bronze import BRONZE_OUTPUT_PATH, RAW_CORPUS_INPUT, load_or_build_bronze_dataframe
from tal_qual.bare_como_spacy_filter import classify_bare_como_candidate
from tal_qual.dataset_expansion_experiments import (
    accepted_candidates_dataframe,
    candidate_counts_dataframe,
    experiment_dataset_path,
    experiment_output_path,
    ground_vehicle_counts_dataframe,
    prefilter_dataset_expansion_bronze_dataframe,
    prepare_dataset_expansion_dataframe,
    review_sample_dataframe,
    vehicle_counts_dataframe,
    write_dataset_expansion_artifacts,
)


## Load Bronze

For full corpus, start Jupyter with:

```bash
TAL_QUAL_RAW_CORPUS_INPUT=data/raw/brwac-clean-*.txt.gz
TAL_QUAL_BRONZE_PATH=data/bronze/brwac_segments_full
```


In [4]:
bronze_path = REPO_ROOT / os.environ.get("TAL_QUAL_BRONZE_PATH", str(BRONZE_OUTPUT_PATH))
raw_path = REPO_ROOT / os.environ.get("TAL_QUAL_RAW_CORPUS_INPUT", str(RAW_CORPUS_INPUT))

bronze_df = load_or_build_bronze_dataframe(spark, raw_path, bronze_path).persist()
print(f"Raw path: {raw_path}")
print(f"Bronze path: {bronze_path}")
print(f"Bronze rows: {bronze_df.count():,}")


Raw path: /home/jovyan/work/data/raw/brwac-clean-1.txt.gz
Bronze path: /home/jovyan/work/data/bronze/brwac_segments
Bronze rows: 4,673,057


## Spark Candidate Discovery


In [5]:
BASE_EXPERIMENT_SLUG = "bare_como_curated_ground_v2"
SPACY_EXPERIMENT_SLUG = "bare_como_spacy_filter"
SPACY_OUTPUT_PATH = Path("outputs/experiments") / SPACY_EXPERIMENT_SLUG
SPACY_DATASET_PATH = Path("data/gold/experiments") / SPACY_EXPERIMENT_SLUG / "candidates"
SPACY_ACCEPTED_PATH = Path("data/gold/experiments") / SPACY_EXPERIMENT_SLUG / "accepted_candidates"

prefiltered_df = prefilter_dataset_expansion_bronze_dataframe(bronze_df, BASE_EXPERIMENT_SLUG).persist()
print(f"Prefiltered bronze rows: {prefiltered_df.count():,}")

candidates_df = prepare_dataset_expansion_dataframe(prefiltered_df, BASE_EXPERIMENT_SLUG).persist()
print(f"Candidate rows: {candidates_df.count():,}")

base_accepted_df = accepted_candidates_dataframe(candidates_df).persist()
print(f"Rule-accepted rows before spaCy: {base_accepted_df.count():,}")


Prefiltered bronze rows: 1,170


/usr/local/spark/python/pyspark/sql/udf.py:134: UserWarning: Cannot infer the eval type from type hints. 
  warnings.warn("Cannot infer the eval type from type hints. ", UserWarning)


Candidate rows: 1,184
Rule-accepted rows before spaCy: 772


## spaCy Classification

This runs spaCy only on the extracted candidate snippets. If the model is not loaded, rebuild/sync the project dependencies inside the Docker image.


In [6]:
nlp = spacy.load("pt_core_news_sm", disable=["ner"])

candidate_pdf = candidates_df.toPandas()
texts = candidate_pdf["candidate_full_text"].fillna("").astype(str).tolist()
records = candidate_pdf.to_dict("records")

nlp_rows = []
for record, doc in zip(records, nlp.pipe(texts, batch_size=500)):
    result = classify_bare_como_candidate(record, doc)
    row = dict(record)
    row.update(result)
    row["pattern_type"] = SPACY_EXPERIMENT_SLUG
    nlp_rows.append(row)

nlp_pdf = pd.DataFrame(nlp_rows)
print(nlp_pdf["nlp_quality_label"].value_counts(dropna=False))
nlp_pdf.head()


nlp_quality_label
reject    510
review    344
keep      330
Name: count, dtype: int64


,candidate_id,pattern_type,source_file,original_line_id,segment_id,connector_text,matched_text,candidate_full_text,text_before,ground_text,...,quality_label,quality_reason,needs_review,char_start,char_end,nlp_quality_label,nlp_quality_reason,spacy_vehicle_head_lemma,spacy_vehicle_head_pos,nlp_needs_review
0,3458664af8593b98ca6df1d80197e5e31c91d8f7,bare_como_spacy_filter,file:///home/jovyan/work/data/raw/brwac-clean-...,281463,2,como,Livre como,Startup Livre: uma iniciativa que visa promove...,Startup Livre: uma iniciativa que visa promove...,Livre,...,keep,short_vehicle,False,69,101,reject,role_head_lexicon,estratégia,NOUN,True
1,6a8b534cb6b92078743be81a8552659634a60913,bare_como_spacy_filter,file:///home/jovyan/work/data/raw/brwac-clean-...,133203,1,como,leve como,Eu queria ser leve como asas de um passarinho,Eu queria ser,leve,...,trimmed,trimmed_clause_like_tail,False,14,34,review,long_vehicle_phrase,asa,NOUN,True
2,70ab7b358b8c3082625a256c04a733010da873b4,bare_como_spacy_filter,file:///home/jovyan/work/data/raw/brwac-clean-...,138737,24,como,alegre como,"Além disso, o samba-enredo, embora muito corre...","Além disso, o samba-enredo, embora muito corre...",alegre,...,reject,bad_vehicle_start,True,68,88,reject,bad_spacy_pos_aux,ser,AUX,True
3,8467d9f71a023b9b77ecef694145b4cfaeb5e3f6,bare_como_spacy_filter,file:///home/jovyan/work/data/raw/brwac-clean-...,18065,22,como,seco como,[Comentários adicionais] É um carro muito conf...,[Comentários adicionais] É um carro muito conf...,seco,...,reject,bad_vehicle_start,True,79,99,reject,bad_spacy_pos_adp,em o,ADP,True
4,a4793932b19ff45581898f437af7343dfebce25e,bare_como_spacy_filter,file:///home/jovyan/work/data/raw/brwac-clean-...,293081,3,como,pequenas como,"bater a obter armas e salvar o seu favorito, c...","bater a obter armas e salvar o seu favorito, c...",pequenas,...,reject,bad_vehicle_start,True,173,208,reject,bad_spacy_pos_verb,poder,VERB,True


## Write NLP-Filtered Artifacts


In [7]:
nlp_df = spark.createDataFrame(nlp_pdf).persist()
nlp_accepted_df = nlp_df.where(F.col("nlp_quality_label").isin("keep", "trimmed")).persist()

print(f"NLP-filtered rows: {nlp_df.count():,}")
print(f"NLP-accepted rows: {nlp_accepted_df.count():,}")

nlp_df.write.mode("overwrite").parquet(str(REPO_ROOT / SPACY_DATASET_PATH))
nlp_accepted_df.write.mode("overwrite").parquet(str(REPO_ROOT / SPACY_ACCEPTED_PATH))

(REPO_ROOT / SPACY_OUTPUT_PATH).mkdir(parents=True, exist_ok=True)
(
    nlp_df.groupBy("pattern_type", "quality_label", "nlp_quality_label", "nlp_quality_reason")
    .count()
    .orderBy(F.col("count").desc(), "quality_label", "nlp_quality_label", "nlp_quality_reason")
    .coalesce(1)
    .write.mode("overwrite")
    .option("header", True)
    .csv(str(REPO_ROOT / SPACY_OUTPUT_PATH / "candidate_counts.csv"))
)
(
    nlp_accepted_df.groupBy("pattern_type", "ground_lemma", "vehicle_head_clean")
    .count()
    .orderBy(F.col("count").desc(), "ground_lemma", "vehicle_head_clean")
    .coalesce(1)
    .write.mode("overwrite")
    .option("header", True)
    .csv(str(REPO_ROOT / SPACY_OUTPUT_PATH / "ground_vehicle_counts.csv"))
)
(
    nlp_accepted_df.groupBy("pattern_type", "vehicle_head_clean", "spacy_vehicle_head_lemma", "spacy_vehicle_head_pos")
    .count()
    .orderBy(F.col("count").desc(), "vehicle_head_clean")
    .coalesce(1)
    .write.mode("overwrite")
    .option("header", True)
    .csv(str(REPO_ROOT / SPACY_OUTPUT_PATH / "vehicle_counts.csv"))
)
(
    nlp_accepted_df.select(
        "candidate_id", "pattern_type", "ground_text", "ground_lemma", "connector_text",
        "vehicle_text_raw", "vehicle_text_clean", "vehicle_head_clean",
        "quality_label", "quality_reason", "nlp_quality_label", "nlp_quality_reason",
        "spacy_vehicle_head_lemma", "spacy_vehicle_head_pos", "candidate_full_text",
    )
    .orderBy("nlp_quality_reason", "ground_lemma", "vehicle_head_clean", "candidate_id")
    .limit(500)
    .coalesce(1)
    .write.mode("overwrite")
    .option("header", True)
    .csv(str(REPO_ROOT / SPACY_OUTPUT_PATH / "accepted_review_sample.csv"))
)
(
    nlp_df.select(
        "candidate_id", "pattern_type", "ground_text", "ground_lemma", "connector_text",
        "vehicle_text_raw", "vehicle_text_clean", "vehicle_head_clean",
        "quality_label", "quality_reason", "nlp_quality_label", "nlp_quality_reason",
        "spacy_vehicle_head_lemma", "spacy_vehicle_head_pos", "candidate_full_text",
    )
    .orderBy(F.col("nlp_needs_review").desc(), "nlp_quality_label", "nlp_quality_reason", "candidate_id")
    .limit(500)
    .coalesce(1)
    .write.mode("overwrite")
    .option("header", True)
    .csv(str(REPO_ROOT / SPACY_OUTPUT_PATH / "review_sample.csv"))
)


NLP-filtered rows: 1,184
NLP-accepted rows: 330


## Inspect


In [8]:
nlp_df.groupBy("quality_label", "nlp_quality_label", "nlp_quality_reason").count().orderBy(F.col("count").desc()).show(80, truncate=False)

nlp_accepted_df.groupBy("ground_lemma", "vehicle_head_clean").count().orderBy(F.col("count").desc(), "ground_lemma", "vehicle_head_clean").show(60, truncate=False)

nlp_accepted_df.select(
    "ground_text", "connector_text", "vehicle_text_clean", "nlp_quality_reason", "spacy_vehicle_head_pos", "candidate_full_text"
).orderBy("nlp_quality_reason", "ground_text", "vehicle_head_clean").show(80, truncate=120)


+-------------+-----------------+---------------------------+-----+
|quality_label|nlp_quality_label|nlp_quality_reason         |count|
+-------------+-----------------+---------------------------+-----+
|trimmed      |review           |long_vehicle_phrase        |237  |
|keep         |keep             |spacy_short_nominal_vehicle|204  |
|reject       |reject           |bad_spacy_pos_pron         |86   |
|reject       |reject           |bad_spacy_pos_adp          |75   |
|reject       |reject           |bad_spacy_pos_det          |58   |
|reject       |reject           |bad_spacy_pos_adv          |51   |
|trimmed      |reject           |bad_spacy_pos_verb         |50   |
|keep         |keep             |visual_head_whitelist      |40   |
|keep         |reject           |bad_spacy_pos_verb         |39   |
|reject       |reject           |bad_spacy_pos_aux          |38   |
|trimmed      |keep             |visual_head_whitelist      |33   |
|reject       |reject           |bad_spacy_pos_s

## Output Index


In [9]:
print(f"full candidates: {SPACY_DATASET_PATH}")
print(f"accepted candidates: {SPACY_ACCEPTED_PATH}")
print(f"artifacts: {SPACY_OUTPUT_PATH}")


full candidates: data/gold/experiments/bare_como_spacy_filter/candidates
accepted candidates: data/gold/experiments/bare_como_spacy_filter/accepted_candidates
artifacts: outputs/experiments/bare_como_spacy_filter
